# 🍎 SentraGrade — DINOv2 SSL Pipeline
## Self-Supervised Feature Extraction + LOCO OOD Evaluation

---
**Dataset**: TR-6 sRGB subset — Banana, Carrot, Guava, Indian Gooseberry, Mango, Tomato  
**Architecture**: DINOv2 ViT-S/14 (frozen backbone → linear probe → full fine-tune)  
**Evaluation**: Leave-One-Class-Out (LOCO) — 6 folds  

---
### 📖 What this notebook does
1. Loads the pre-trained & fine-tuned DINOv2 checkpoints
2. Runs inference and extracts L2-normalised embeddings for train / val / OOD splits
3. Generates: Confusion Matrix · Training Curves · t-SNE · UMAP
4. Computes OOD scores: Energy · MSP · Prototype Distance
5. Produces a cross-fold AUROC summary table


## 1. Environment Setup

In [1]:
import sys

# ── Detect Google Colab ───────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab — mounting Drive...")
    from google.colab import drive
    drive.mount("/content/drive")

# ── Install required packages ─────────────────────────────────────────────────
import subprocess
pkgs = ["timm", "umap-learn", "seaborn", "pandas", "scikit-learn", "Pillow", "tqdm"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                   capture_output=True)
print("✓ Packages ready")

✓ Packages ready


## 2. Imports

In [2]:
import os, json, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

import matplotlib
matplotlib.use("Agg")  # headless-safe; change to 'inline' if plots don't show
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    f1_score, roc_auc_score, precision_score, recall_score,
)
import timm

try:
    import umap as umap_lib
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠ umap-learn not installed — UMAP plots will be skipped")
    print("  Install with: pip install umap-learn")

print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")
print(f"UMAP     : {'available' if UMAP_AVAILABLE else 'not installed'}")

PyTorch  : 2.8.0
timm     : 1.0.28
UMAP     : available


## 3. Configuration


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         CHANGE THESE TWO PATHS FOR YOUR MACHINE                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# Path to the sentragrade_data/ folder (contains folds/ and resized/)
DATA_ROOT = Path("/Users/nehan/Desktop/Semi-Supervised/sentragrade_data")

# Path to the project/checkpoints/ folder (contains best_model_foldX.pt)
CHECKPOINT_DIR = Path("/Users/nehan/Desktop/Semi-Supervised/project/checkpoints")

# ── Google Colab example (uncomment if using Colab) ──────────────────────────
# DATA_ROOT      = Path("/content/drive/MyDrive/sentragrade_data")
# CHECKPOINT_DIR = Path("/content/drive/MyDrive/project/checkpoints")

# ═════════════════════════════════════════════════════════════════════════════
# Settings — no need to change
# ═════════════════════════════════════════════════════════════════════════════
FOLDS_DIR    = DATA_ROOT / "folds"
RESIZED_ROOT = DATA_ROOT / "resized"

IMAGE_SIZE     = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
NUM_FOLDS      = 6
EMBED_DIM      = 384          # DINOv2 ViT-S/14
TIMM_MODEL     = "vit_small_patch14_dinov2.lvd142m"

# Output directory for plots and embeddings
OUTPUT_DIR = Path("./sentragrade_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMBED_OUT  = OUTPUT_DIR / "embeddings"
EMBED_OUT.mkdir(exist_ok=True)
PLOT_OUT   = OUTPUT_DIR / "plots"
PLOT_OUT.mkdir(exist_ok=True)

# Path remapping — the fold CSVs have hardcoded paths from original machine
_OLD_PREFIX = "/Users/riteeshtm/sentragrade_data/resized"
def remap_path(csv_path: str) -> str:
    p = str(csv_path)
    if os.path.isfile(p):
        return p
    if p.startswith(_OLD_PREFIX):
        rel = p[len(_OLD_PREFIX):].lstrip("/")
        remapped = str(RESIZED_ROOT / rel)
        if os.path.isfile(remapped):
            return remapped
    return str(RESIZED_ROOT / Path(p).parent.name / Path(p).name)

# Device
def get_device():
    if torch.cuda.is_available():              return torch.device("cuda")
    if torch.backends.mps.is_available():      return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Device          : {DEVICE}")
print(f"Data root       : {DATA_ROOT}  (exists={DATA_ROOT.exists()})")
print(f"Checkpoints     : {CHECKPOINT_DIR}  (exists={CHECKPOINT_DIR.exists()})")
print(f"Output dir      : {OUTPUT_DIR}")

Device          : mps
Data root       : /Users/nehan/Desktop/Semi-Supervised/sentragrade_data  (exists=True)
Checkpoints     : /Users/nehan/Desktop/Semi-Supervised/project/checkpoints  (exists=True)
Output dir      : sentragrade_outputs


## 4. Dataset Exploration

In [4]:
# ── Fold summary ──────────────────────────────────────────────────────────────
fold_summary = pd.read_csv(FOLDS_DIR / "fold_summary.csv")
print("LOCO Fold Summary")
print("="*50)
display(fold_summary)
print()

# ── Per-fold class distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
colors = plt.get_cmap("tab10").colors

for fold in range(NUM_FOLDS):
    df = pd.read_csv(FOLDS_DIR / f"fold{fold}_train.csv")
    counts = df["class_name"].value_counts().sort_index()
    ax = axes[fold]
    bars = ax.bar(counts.index, counts.values,
                  color=[colors[i % 10] for i in range(len(counts))])
    ax.set_title(f"Fold {fold}  (OOD: {fold_summary.loc[fold, 'held_out_class']})",
                 fontsize=11)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(val), ha="center", va="bottom", fontsize=8)

plt.suptitle("Training Set Class Distribution per Fold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_OUT / "class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {PLOT_OUT / 'class_distribution.png'}")

LOCO Fold Summary


,fold,held_out_class,train_n,val_n,ood_n
0,0,Banana,3978,1037,323
1,1,Carrot,3843,1037,458
2,2,Guava,3791,1037,510
3,3,Indian_Gooseberry,3077,850,1411
4,4,Mango,3892,1054,392
5,5,Tomato,2414,680,2244



Saved → sentragrade_outputs/plots/class_distribution.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/2405208508.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Sample images grid ────────────────────────────────────────────────────────
ALL_CLASSES = ["Banana", "Carrot", "Guava", "Indian_Gooseberry", "Mango", "Tomato"]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, cls in enumerate(ALL_CLASSES):
    cls_dir = RESIZED_ROOT / cls
    imgs = sorted(cls_dir.glob("*.jpg"))[:1]
    if imgs:
        img = Image.open(imgs[0]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(cls, fontsize=12, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Sample Image per Class", fontsize=14)
plt.tight_layout()
plt.savefig(PLOT_OUT / "sample_images.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {PLOT_OUT / 'sample_images.png'}")

Saved → sentragrade_outputs/plots/sample_images.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/2151256732.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Model Architecture

In [6]:
class DINOv2Backbone(nn.Module):
    """
    DINOv2 ViT-S/14 backbone loaded via timm.
    With num_classes=0 and img_size=224, forward() returns (B, 384).
    """
    def __init__(self, frozen=True):
        super().__init__()
        print(f"Loading {TIMM_MODEL} via timm...")
        self.backbone = timm.create_model(
            TIMM_MODEL, pretrained=True, num_classes=0, img_size=IMAGE_SIZE
        )
        self.embed_dim = EMBED_DIM
        if frozen:
            for p in self.backbone.parameters():
                p.requires_grad = False
            self.backbone.eval()
            print("✓ Backbone frozen (linear-probe mode)")
        else:
            print("✓ Backbone unfrozen (fine-tune mode)")

    def forward(self, x):
        return self.backbone(x)


class LinearProbeClassifier(nn.Module):
    """Frozen DINOv2 + trainable linear head."""
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone    = backbone
        self.embed_dim   = backbone.embed_dim
        self.num_classes = num_classes
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(p=dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x)
        return self.head(feats)

    @torch.no_grad()
    def get_embeddings(self, x):
        return F.normalize(self.backbone(x), p=2, dim=1)


class FineTunedClassifier(nn.Module):
    """Unfrozen DINOv2 + linear head (inherits from probe model)."""
    def __init__(self, probe_model):
        super().__init__()
        self.backbone    = probe_model.backbone
        self.head        = probe_model.head
        self.embed_dim   = probe_model.embed_dim
        self.num_classes = probe_model.num_classes
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))

    @torch.no_grad()
    def get_embeddings(self, x):
        return F.normalize(self.backbone(x), p=2, dim=1)


print("Model classes defined ✓")

Model classes defined ✓


## 6. Load Trained Checkpoints

In [7]:
# ── Check which checkpoints exist ─────────────────────────────────────────────
print("Checking checkpoints...")
print("-" * 50)
for fold in range(NUM_FOLDS):
    ft_ckpt = CHECKPOINT_DIR / f"best_model_fold{fold}.pt"
    pr_ckpt = CHECKPOINT_DIR / f"best_model_fold{fold}_probe.pt"
    status = "✅ fine-tuned" if ft_ckpt.exists() else ("🟡 probe only" if pr_ckpt.exists() else "❌ missing")
    print(f"  Fold {fold}: {status}")

Checking checkpoints...
--------------------------------------------------
  Fold 0: ✅ fine-tuned
  Fold 1: ✅ fine-tuned
  Fold 2: ✅ fine-tuned
  Fold 3: ✅ fine-tuned
  Fold 4: ✅ fine-tuned
  Fold 5: ✅ fine-tuned


In [8]:
def load_model_for_fold(fold: int, device: torch.device):
    """
    Load the best available model for the given fold.
    Prefers fine-tuned > probe checkpoint.
    Returns (model, label_map, num_classes).
    """
    ft_path  = CHECKPOINT_DIR / f"best_model_fold{fold}.pt"
    pr_path  = CHECKPOINT_DIR / f"best_model_fold{fold}_probe.pt"
    ckpt_path = ft_path if ft_path.exists() else pr_path

    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"No checkpoint found for fold {fold}. "
            "Run train_dinov2.py first."
        )

    ckpt        = torch.load(ckpt_path, map_location=device, weights_only=False)
    label_map   = ckpt["label_map"]
    num_classes = ckpt["num_classes"]
    stage       = ckpt.get("stage", "probe")

    backbone = DINOv2Backbone(frozen=True)
    probe    = LinearProbeClassifier(backbone, num_classes=num_classes)

    if stage == "finetune":
        model = FineTunedClassifier(probe)
    else:
        model = probe

    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device).eval()

    print(f"Fold {fold} [{stage}] loaded  |  "
          f"label_map={label_map}  |  "
          f"best_f1={ckpt.get('best_f1', 'N/A'):.4f}")
    return model, label_map, num_classes


# Quick test — load fold 0
model0, lmap0, nc0 = load_model_for_fold(0, DEVICE)
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
with torch.no_grad():
    logits = model0(dummy)
    embeds = model0.get_embeddings(dummy)
print(f"\nLogits shape    : {logits.shape}")
print(f"Embeddings shape: {embeds.shape}")
print(f"Embed norms     : {embeds.norm(dim=1).tolist()}")

Loading vit_small_patch14_dinov2.lvd142m via timm...


✓ Backbone frozen (linear-probe mode)
Fold 0 [finetune] loaded  |  label_map={'Carrot': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000

Logits shape    : torch.Size([2, 5])
Embeddings shape: torch.Size([2, 384])
Embed norms     : [1.0, 1.0]


## 7. Extract Embeddings for All Folds

In [9]:
# ── Transforms ────────────────────────────────────────────────────────────────
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class FoldDataset(Dataset):
    """Simple dataset that reads a fold CSV and returns (image_tensor, label)."""
    def __init__(self, csv_path, label_map, transform, is_ood=False):
        df   = pd.read_csv(csv_path)
        self.transform  = transform
        self.is_ood     = is_ood
        self.label_map  = label_map
        self.samples    = []
        skipped = 0
        for _, row in df.iterrows():
            p  = remap_path(row["resized_path"])
            cn = str(row["class_name"])
            if not os.path.isfile(p):
                skipped += 1
                continue
            lbl = -1 if is_ood else label_map.get(cn, -1)
            self.samples.append((p, cn, lbl))
        if skipped:
            print(f"  ⚠ {skipped} images skipped (file not found)")

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, cn, lbl = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), lbl


@torch.no_grad()
def extract_embeddings(model, loader, device):
    """Return (embeddings_np, labels_np) for all items in loader."""
    model.eval()
    all_emb, all_lbl = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        emb  = model.get_embeddings(imgs).cpu().numpy()
        all_emb.append(emb)
        all_lbl.extend(lbls.tolist())
    return np.vstack(all_emb).astype(np.float32), np.array(all_lbl, dtype=np.int32)


print("Dataset and embedding utilities defined ✓")

Dataset and embedding utilities defined ✓


In [10]:
# ── Extract and save embeddings for every fold ────────────────────────────────
# This takes ~5-10 min total (each forward pass is fast on MPS/CUDA)
# Already-saved .npy files are loaded directly to save time.

fold_embeddings = {}   # fold → {train, val, ood} → {emb, lbl, class_names}

for fold in range(NUM_FOLDS):
    print(f"\n{'='*50}")
    print(f"Fold {fold}")
    print(f"{'='*50}")

    # Load label map
    with open(FOLDS_DIR / f"fold{fold}_label_map.json") as f:
        label_map = json.load(f)
    class_names = sorted(label_map, key=label_map.get)
    num_classes = len(label_map)

    # Load model
    model, _, _ = load_model_for_fold(fold, DEVICE)
    fold_embeddings[fold] = {"class_names": class_names, "num_classes": num_classes}

    for split, is_ood in [("train", False), ("val", False), ("ood", True)]:
        out_emb = EMBED_OUT / f"fold{fold}_{split}_embeddings.npy"
        out_lbl = EMBED_OUT / f"fold{fold}_{split}_labels.npy"

        if out_emb.exists() and out_lbl.exists():
            emb = np.load(out_emb)
            lbl = np.load(out_lbl)
            print(f"  [{split}] Loaded from cache  shape={emb.shape}")
        else:
            ds = FoldDataset(
                FOLDS_DIR / f"fold{fold}_{split}.csv",
                label_map, val_transform, is_ood=is_ood
            )
            ld = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
            print(f"  [{split}] Extracting {len(ds)} samples...", end="", flush=True)
            emb, lbl = extract_embeddings(model, ld, DEVICE)
            np.save(out_emb, emb)
            np.save(out_lbl, lbl)
            print(f"  shape={emb.shape}  ✓")

        fold_embeddings[fold][split] = {"emb": emb, "lbl": lbl}

    # Free GPU/MPS memory
    del model
    if DEVICE.type in ("cuda", "mps"):
        torch.mps.empty_cache() if DEVICE.type == "mps" else torch.cuda.empty_cache()

print("\n✓ All embeddings extracted and cached.")


Fold 0
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 0 [finetune] loaded  |  label_map={'Carrot': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
  [train] Extracting 3978 samples...  shape=(3978, 384)  ✓
  [val] Extracting 1037 samples...  shape=(1037, 384)  ✓
  [ood] Extracting 323 samples...  shape=(323, 384)  ✓

Fold 1
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 1 [finetune] loaded  |  label_map={'Banana': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
  [train] Extracting 3843 samples...  shape=(3843, 384)  ✓
  [val] Extracting 1037 samples...  shape=(1037, 384)  ✓
  [ood] Extracting 458 samples...  shape=(458, 384)  ✓

Fold 2
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 2 [finetune] loaded  |  label_map={'Banana': 0, 'Carrot': 1, 'Indian_Gooseberry': 2, 'Mang

## 8. Visualizations
### 8.1 Confusion Matrices (Validation Set)

In [11]:
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()

fold_f1s = []
for fold in range(NUM_FOLDS):
    model, label_map, num_classes = load_model_for_fold(fold, DEVICE)
    class_names = sorted(label_map, key=label_map.get)

    ds = FoldDataset(FOLDS_DIR / f"fold{fold}_val.csv", label_map, val_transform)
    ld = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for imgs, lbls in ld:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(dim=1).cpu().tolist()
            y_pred.extend(preds)
            y_true.extend(lbls.tolist())

    f1  = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred)
    fold_f1s.append(f1)

    ax = axes[fold]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"Fold {fold}  (OOD: {fold_summary.loc[fold,'held_out_class']})\n"
                 f"Acc={acc:.4f}  F1={f1:.4f}", fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=30)

    del model

plt.suptitle("Confusion Matrices — Validation Set (All Folds)", fontsize=14)
plt.tight_layout()
plt.savefig(PLOT_OUT / "confusion_matrices.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"\nMean Val F1 across folds: {np.mean(fold_f1s):.4f}")
print(f"Saved → {PLOT_OUT / 'confusion_matrices.png'}")

Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 0 [finetune] loaded  |  label_map={'Carrot': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 1 [finetune] loaded  |  label_map={'Banana': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 2 [finetune] loaded  |  label_map={'Banana': 0, 'Carrot': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 3 [finetune] loaded  |  label_map={'Banana': 0, 'Carrot': 1, 'Guava': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 4 [finetune] loade

/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/2625109639.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.2 t-SNE: In-Distribution Train + OOD

In [12]:
from sklearn.manifold import TSNE
import matplotlib.cm as cm

PALETTE = plt.get_cmap("tab10").colors
N_TSNE  = 1500   # max points per fold (faster)

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()

for fold in range(NUM_FOLDS):
    class_names = fold_embeddings[fold]["class_names"]
    tr_emb = fold_embeddings[fold]["train"]["emb"]
    tr_lbl = fold_embeddings[fold]["train"]["lbl"]
    od_emb = fold_embeddings[fold]["ood"]["emb"]

    rng    = np.random.RandomState(42)
    n_tr   = min(len(tr_emb), N_TSNE // 2)
    n_od   = min(len(od_emb), N_TSNE - n_tr)
    tr_idx = rng.choice(len(tr_emb), n_tr, replace=False)
    od_idx = rng.choice(len(od_emb), n_od, replace=False)

    combined = np.vstack([tr_emb[tr_idx], od_emb[od_idx]])
    labels   = np.concatenate([tr_lbl[tr_idx], np.full(n_od, -1)])
    is_ood   = np.array([False]*n_tr + [True]*n_od)

    xy = TSNE(n_components=2, perplexity=30, random_state=42,
              init="pca", learning_rate="auto").fit_transform(combined)

    ax = axes[fold]
    for ci, cn in enumerate(class_names):
        mask = (~is_ood) & (labels == ci)
        if mask.any():
            ax.scatter(xy[mask, 0], xy[mask, 1], c=[PALETTE[ci % 10]],
                       label=cn, alpha=0.6, s=12, linewidths=0)
    if is_ood.any():
        ax.scatter(xy[is_ood, 0], xy[is_ood, 1], c="lightgrey",
                   marker="x", label=f"OOD ({fold_summary.loc[fold,'held_out_class']})",
                   alpha=0.5, s=20, linewidths=0.8)

    ax.set_title(f"Fold {fold}  — t-SNE", fontsize=10)
    ax.legend(fontsize=7, markerscale=1.5, framealpha=0.7)
    ax.grid(True, alpha=0.2)

plt.suptitle("t-SNE: In-Distribution (coloured) vs OOD (grey ×)", fontsize=14)
plt.tight_layout()
plt.savefig(PLOT_OUT / "tsne_all_folds.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {PLOT_OUT / 'tsne_all_folds.png'}")

Saved → sentragrade_outputs/plots/tsne_all_folds.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/4269567115.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.3 UMAP: In-Distribution Train + OOD

In [13]:
if not UMAP_AVAILABLE:
    print("umap-learn not installed — skipping UMAP. pip install umap-learn")
else:
    N_UMAP = 2000

    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()

    for fold in range(NUM_FOLDS):
        class_names = fold_embeddings[fold]["class_names"]
        tr_emb = fold_embeddings[fold]["train"]["emb"]
        tr_lbl = fold_embeddings[fold]["train"]["lbl"]
        od_emb = fold_embeddings[fold]["ood"]["emb"]

        rng    = np.random.RandomState(42)
        n_tr   = min(len(tr_emb), N_UMAP // 2)
        n_od   = min(len(od_emb), N_UMAP - n_tr)
        tr_idx = rng.choice(len(tr_emb), n_tr, replace=False)
        od_idx = rng.choice(len(od_emb), n_od, replace=False)

        combined = np.vstack([tr_emb[tr_idx], od_emb[od_idx]])
        labels   = np.concatenate([tr_lbl[tr_idx], np.full(n_od, -1)])
        is_ood   = np.array([False]*n_tr + [True]*n_od)

        reducer = umap_lib.UMAP(n_components=2, n_neighbors=15,
                                min_dist=0.1, metric="cosine", random_state=42)
        xy = reducer.fit_transform(combined)

        ax = axes[fold]
        for ci, cn in enumerate(class_names):
            mask = (~is_ood) & (labels == ci)
            if mask.any():
                ax.scatter(xy[mask, 0], xy[mask, 1], c=[PALETTE[ci % 10]],
                           label=cn, alpha=0.6, s=12, linewidths=0)
        if is_ood.any():
            ax.scatter(xy[is_ood, 0], xy[is_ood, 1], c="lightgrey",
                       marker="x", label=f"OOD ({fold_summary.loc[fold,'held_out_class']})",
                       alpha=0.5, s=20, linewidths=0.8)

        ax.set_title(f"Fold {fold}  — UMAP", fontsize=10)
        ax.legend(fontsize=7, markerscale=1.5, framealpha=0.7)
        ax.grid(True, alpha=0.2)

    plt.suptitle("UMAP: In-Distribution (coloured) vs OOD (grey ×)", fontsize=14)
    plt.tight_layout()
    plt.savefig(PLOT_OUT / "umap_all_folds.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved → {PLOT_OUT / 'umap_all_folds.png'}")

/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nehan/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarn

Saved → sentragrade_outputs/plots/umap_all_folds.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/2287605002.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. OOD Score Analysis

Three scoring methods — all use the trained model + val/OOD embeddings/logits:

| Method | Formula | Higher = more OOD? |
|--------|---------|-------------------|
| **Energy** | −log Σ exp(logit_c) | Yes — ID has lower energy |
| **MSP** | max softmax probability | No — ID has higher MSP |
| **Prototype Distance** | min distance to class centroid (embedding space) | Yes — OOD is farther from all centroids |


In [14]:
@torch.no_grad()
def compute_ood_scores(model, loader, device):
    """
    Returns:
        energy_scores   : (N,)  − lower = more in-distribution
        msp_scores      : (N,)  − higher = more in-distribution
        logits_all      : (N, C)
    """
    model.eval()
    all_logits = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu().float())
    all_logits  = torch.cat(all_logits, dim=0)          # (N, C)
    energy      = -torch.logsumexp(all_logits, dim=1).numpy()   # (N,)
    msp         = torch.softmax(all_logits, dim=1).max(dim=1).values.numpy()
    return energy, msp, all_logits.numpy()


def prototype_distances(train_emb, train_lbl, query_emb, num_classes):
    """
    Compute distance from each query to the nearest class centroid.
    Returns (N,) — higher = more OOD.
    """
    centroids = np.stack([
        train_emb[train_lbl == c].mean(axis=0)
        for c in range(num_classes)
    ])                                                    # (C, D)
    # Cosine distance: 1 - cosine_similarity
    c_norm = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-8)
    q_norm = query_emb / (np.linalg.norm(query_emb, axis=1, keepdims=True) + 1e-8)
    sims   = q_norm @ c_norm.T                            # (N, C)
    return 1.0 - sims.max(axis=1)                         # min distance to nearest centroid


print("OOD scoring utilities defined ✓")

OOD scoring utilities defined ✓


In [15]:
# ── Compute OOD scores for every fold ─────────────────────────────────────────
# AUROC: area under ROC using score to separate ID-val from OOD
results = []   # list of dicts for summary table

for fold in range(NUM_FOLDS):
    print(f"\nFold {fold}", "─"*40)
    model, label_map, num_classes = load_model_for_fold(fold, DEVICE)
    class_names = sorted(label_map, key=label_map.get)
    held_out    = fold_summary.loc[fold, "held_out_class"]

    # Build val and OOD loaders
    val_ds  = FoldDataset(FOLDS_DIR / f"fold{fold}_val.csv", label_map, val_transform)
    ood_ds  = FoldDataset(FOLDS_DIR / f"fold{fold}_ood.csv", label_map, val_transform, is_ood=True)
    val_ld  = DataLoader(val_ds,  batch_size=64, shuffle=False, num_workers=0)
    ood_ld  = DataLoader(ood_ds,  batch_size=64, shuffle=False, num_workers=0)

    # Energy & MSP
    val_energy, val_msp, val_logits = compute_ood_scores(model, val_ld, DEVICE)
    ood_energy, ood_msp, ood_logits = compute_ood_scores(model, ood_ld, DEVICE)

    # Prototype distance
    tr_emb  = fold_embeddings[fold]["train"]["emb"]
    tr_lbl  = fold_embeddings[fold]["train"]["lbl"]
    val_emb = fold_embeddings[fold]["val"]["emb"]
    ood_emb = fold_embeddings[fold]["ood"]["emb"]

    val_proto = prototype_distances(tr_emb, tr_lbl, val_emb, num_classes)
    ood_proto = prototype_distances(tr_emb, tr_lbl, ood_emb, num_classes)

    # AUROC (binary: ID=0, OOD=1)
    # For energy/proto: higher = OOD  → labels=[0]*N_val + [1]*N_ood, scores=combined
    # For MSP:          lower  = OOD  → negate
    n_val, n_ood = len(val_energy), len(ood_energy)
    labels_bin   = np.array([0]*n_val + [1]*n_ood)

    auroc_energy = roc_auc_score(labels_bin, np.concatenate([val_energy, ood_energy]))
    auroc_msp    = roc_auc_score(labels_bin, np.concatenate([-val_msp,  -ood_msp ]))
    auroc_proto  = roc_auc_score(labels_bin, np.concatenate([val_proto,  ood_proto]))

    print(f"  AUROC Energy   : {auroc_energy:.4f}")
    print(f"  AUROC MSP      : {auroc_msp:.4f}")
    print(f"  AUROC Prototype: {auroc_proto:.4f}")

    results.append({
        "Fold":              fold,
        "OOD Class":         held_out,
        "N_val":             n_val,
        "N_ood":             n_ood,
        "AUROC Energy":      round(auroc_energy, 4),
        "AUROC MSP":         round(auroc_msp,    4),
        "AUROC Prototype":   round(auroc_proto,   4),
    })

    del model

print("\n✓ OOD scoring complete")


Fold 0 ────────────────────────────────────────
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 0 [finetune] loaded  |  label_map={'Carrot': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
  AUROC Energy   : 1.0000
  AUROC MSP      : 1.0000
  AUROC Prototype: 1.0000

Fold 1 ────────────────────────────────────────
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 1 [finetune] loaded  |  label_map={'Banana': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
  AUROC Energy   : 0.9870
  AUROC MSP      : 1.0000
  AUROC Prototype: 1.0000

Fold 2 ────────────────────────────────────────
Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 2 [finetune] loaded  |  label_map={'Banana': 0, 'Carrot': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
  AUROC Energy   : 0.999

In [16]:
# ── OOD Score Histograms (fold 0 example) ─────────────────────────────────────
fold = 0
model, label_map, num_classes = load_model_for_fold(fold, DEVICE)

val_ds = FoldDataset(FOLDS_DIR / f"fold{fold}_val.csv", label_map, val_transform)
ood_ds = FoldDataset(FOLDS_DIR / f"fold{fold}_ood.csv", label_map, val_transform, is_ood=True)
val_ld = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)
ood_ld = DataLoader(ood_ds, batch_size=64, shuffle=False, num_workers=0)

val_energy, val_msp, _ = compute_ood_scores(model, val_ld, DEVICE)
ood_energy, ood_msp, _ = compute_ood_scores(model, ood_ld, DEVICE)

val_emb  = fold_embeddings[fold]["val"]["emb"]
ood_emb  = fold_embeddings[fold]["ood"]["emb"]
tr_emb   = fold_embeddings[fold]["train"]["emb"]
tr_lbl   = fold_embeddings[fold]["train"]["lbl"]
val_proto = prototype_distances(tr_emb, tr_lbl, val_emb, num_classes)
ood_proto = prototype_distances(tr_emb, tr_lbl, ood_emb, num_classes)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ood_class = fold_summary.loc[fold, "held_out_class"]

for ax, (id_s, ood_s, title, xlabel) in zip(axes, [
    (val_energy, ood_energy, "Energy Score",        "−log Σ exp(logit)"),
    (-val_msp,  -ood_msp,   "MSP Score (negated)", "−max softmax prob"),
    (val_proto,  ood_proto,  "Prototype Distance",  "1 − max cosine sim"),
]):
    ax.hist(id_s,  bins=50, alpha=0.7, color="#2196F3", label="ID (val)")
    ax.hist(ood_s, bins=50, alpha=0.7, color="#FF5722", label=f"OOD ({ood_class})")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"OOD Score Distributions — Fold {fold}", fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_OUT / f"fold{fold}_ood_scores.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {PLOT_OUT / f'fold{fold}_ood_scores.png'}")
del model

Loading vit_small_patch14_dinov2.lvd142m via timm...
✓ Backbone frozen (linear-probe mode)
Fold 0 [finetune] loaded  |  label_map={'Carrot': 0, 'Guava': 1, 'Indian_Gooseberry': 2, 'Mango': 3, 'Tomato': 4}  |  best_f1=1.0000
Saved → sentragrade_outputs/plots/fold0_ood_scores.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/932367135.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Cross-Fold Results Summary

In [17]:
results_df = pd.DataFrame(results)

# Add mean row
mean_row = {
    "Fold":            "Mean",
    "OOD Class":       "—",
    "N_val":           int(results_df["N_val"].mean()),
    "N_ood":           int(results_df["N_ood"].mean()),
    "AUROC Energy":    round(results_df["AUROC Energy"].mean(), 4),
    "AUROC MSP":       round(results_df["AUROC MSP"].mean(), 4),
    "AUROC Prototype": round(results_df["AUROC Prototype"].mean(), 4),
}
results_df = pd.concat([results_df, pd.DataFrame([mean_row])], ignore_index=True)

print("SentraGrade — DINOv2 LOCO OOD Results")
print("="*70)
display(results_df.style
    .format({"AUROC Energy": "{:.4f}", "AUROC MSP": "{:.4f}", "AUROC Prototype": "{:.4f}"})
    .background_gradient(subset=["AUROC Energy","AUROC MSP","AUROC Prototype"], cmap="RdYlGn")
)

results_df.to_csv(OUTPUT_DIR / "ood_results_summary.csv", index=False)
print(f"\nSaved → {OUTPUT_DIR / 'ood_results_summary.csv'}")

SentraGrade — DINOv2 LOCO OOD Results


,Fold,OOD Class,N_val,N_ood,AUROC Energy,AUROC MSP,AUROC Prototype
0,0,Banana,1037,323,1.0000,1.0000,1.0000
1,1,Carrot,1037,458,0.9870,1.0000,1.0000
2,2,Guava,1037,510,0.9996,0.9993,0.9999
3,3,Indian_Gooseberry,850,1411,0.9937,0.9953,0.9989
4,4,Mango,1054,392,1.0000,1.0000,0.9999
5,5,Tomato,680,2244,1.0000,1.0000,1.0000
6,Mean,—,949,889,0.9967,0.9991,0.9998



Saved → sentragrade_outputs/ood_results_summary.csv


In [18]:
# ── Bar chart of AUROC per fold ───────────────────────────────────────────────
df_plot = results_df[results_df["Fold"] != "Mean"].copy()
df_plot["Fold"] = df_plot["Fold"].astype(int)

x = np.arange(NUM_FOLDS)
w = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, df_plot["AUROC Energy"].values,    w, label="Energy",    color="#2196F3")
ax.bar(x,     df_plot["AUROC MSP"].values,        w, label="MSP",       color="#4CAF50")
ax.bar(x + w, df_plot["AUROC Prototype"].values,  w, label="Prototype", color="#FF9800")

ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i}\n(OOD: {r})"
                    for i, r in zip(df_plot["Fold"], df_plot["OOD Class"])],
                   fontsize=9)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="Random (0.5)")
ax.set_ylabel("AUROC")
ax.set_title("OOD Detection AUROC — DINOv2 ViT-S/14 (LVD-142M)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_OUT / "ood_auroc_bar.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {PLOT_OUT / 'ood_auroc_bar.png'}")

Saved → sentragrade_outputs/plots/ood_auroc_bar.png


/var/folders/j6/mnb27kv572775mqzs___dmq40000gn/T/ipykernel_1515/427603144.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Summary & Next Steps

### ✅ What this notebook produced
- **Embeddings**: `sentragrade_outputs/embeddings/fold{0-5}_{train,val,ood}_embeddings.npy`
- **Plots**: `sentragrade_outputs/plots/` — confusion matrices, t-SNE, UMAP, OOD histograms, AUROC bars
- **Results**: `sentragrade_outputs/ood_results_summary.csv`

### 🔬 Interpretation
- **Val F1 = 1.0000 on all folds** — DINOv2's LVD-142M pretrained features perfectly separate the 5 fruit/vegetable classes.
- **AUROC values** show how well each OOD scoring method identifies the held-out class.
- Classes with distinctive visual features (Banana, Carrot) will have higher AUROC than visually similar ones.

### 🚀 Next Steps (for teammate)
1. **Energy-based OOD threshold**: Use val energy distribution to set a decision boundary
2. **Prototype Nearest Neighbour**: Replace linear probe with kNN on train embeddings
3. **Feature ensemble**: Combine Energy + Prototype scores for better AUROC
4. **Calibration**: Temperature scaling on logits to improve MSP reliability
5. **Grade prediction**: Use the embedding space for downstream produce quality grading
